# Querying RSEc dumps

## Loading RDF data

### Loading RSEc dumps into an RDFlib graph

In [2]:
!pip install rdflib
!pip install pyyaml


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
from rdflib import Graph, Literal
#import rdflib
import os

In [4]:
#edam_version = 'https://edamontology.org/EDAM_1.25.owl'
#edam_version = 'https://raw.githubusercontent.com/edamontology/edamontology/master/EDAM_dev.owl'
#edam = 'https://github.com/edamontology/edamontology/raw/main/EDAM_dev.owl'

bioconda_rdf = 'https://raw.githubusercontent.com/rioualen/RSEc-SPARQL/refs/heads/main/data/bioconda-dump.ttl'
biocontainers_rdf = 'https://raw.githubusercontent.com/rioualen/RSEc-SPARQL/refs/heads/main/data/biocontainers-dump.ttl'
bioschemas_rdf = 'https://raw.githubusercontent.com/rioualen/RSEc-SPARQL/refs/heads/main/data/bioschemas-dump.ttl'
debian_rdf = 'https://raw.githubusercontent.com/rioualen/RSEc-SPARQL/refs/heads/main/data/debian-dump.ttl'
workflowhub_rdf = 'https://raw.githubusercontent.com/rioualen/RSEc-SPARQL/refs/heads/main/data/workflow_hub.ttl'

kg = Graph()
kg.parse(bioconda_rdf)
kg.parse(biocontainers_rdf)
kg.parse(bioschemas_rdf)
kg.parse(debian_rdf)
kg.parse(workflowhub_rdf)

print(str(len(kg)) + ' triples in the kg after parsing all datasets')


787060 triples in the kg after parsing all datasets


### Loading EDAM and adding to graph

In [5]:
#edam_version = 'https://edamontology.org/EDAM_1.25.owl'
#edam_version = 'https://raw.githubusercontent.com/edamontology/edamontology/master/EDAM_dev.owl'
edam = 'https://github.com/edamontology/edamontology/raw/main/EDAM_dev.owl'

edam_kg = Graph()
edam_kg.parse(edam, format='xml')

print(str(len(edam_kg)) + ' triples in the edam_kg after parsing all datasets')


kg.parse(edam, format='xml')
print(str(len(kg)) + ' triples in the kg after parsing all datasets')


39057 triples in the edam_kg after parsing all datasets
826117 triples in the kg after parsing all datasets


## SPARQL queries 

### How many SoftwareApplication entries

In [139]:
query = """
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT (COUNT(?s) as ?count) WHERE {
    ?s rdf:type schema:SoftwareApplication .
}
"""
results = kg.query(query)

for r in results :
#      print(r)
    print(f"{r['count']}")


42804


### SoftwareApplication and biotools IDs

In [ ]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?application ?biotools_ID 
WHERE {
    ?application schema:identifier ?biotools_ID .
}
ORDER BY ?biotools_ID
LIMIT 50

"""
results = kg.query(query)

for r in results :
    print(f"{r['application']} - {r['biotools_ID']}")


https://bioconda.github.io/recipes/abricate - https://bio.tools/ABRicate
https://bioconda.github.io/recipes/admixtools - https://bio.tools/AdmixTools
https://biocontainers.pro/tools/admixtools - https://bio.tools/AdmixTools
https://bioconda.github.io/recipes/arriba - https://bio.tools/Arriba
https://bioconda.github.io/recipes/augur - https://bio.tools/Augur
https://bioconda.github.io/recipes/beagle - https://bio.tools/BEAGLE
https://biocontainers.pro/tools/beagle - https://bio.tools/BEAGLE
https://bioconda.github.io/recipes/bufet - https://bio.tools/BUFET
https://biocontainers.pro/tools/bufet - https://bio.tools/BUFET
https://bioconda.github.io/recipes/bamutil - https://bio.tools/Bamutil
https://biocontainers.pro/tools/bamutil - https://bio.tools/Bamutil
https://bioconda.github.io/recipes/bedops - https://bio.tools/Bedops
https://biocontainers.pro/tools/bedops - https://bio.tools/Bedops
https://bioconda.github.io/recipes/bioconductor-biocworkflowtools - https://bio.tools/BiocWorkflowTo

### Entries per biotools ID

In [116]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?biotools_ID (COUNT (?application) AS ?count)
WHERE {
    ?application schema:identifier ?biotools_ID .
}
GROUP BY ?biotools_ID
ORDER BY DESC(?count)
LIMIT 10
"""
results = kg.query(query)

for r in results :
    print(f"{r['biotools_ID']} - {r['count']}")

https://bio.tools/UCSC_Genome_Browser_Utilities - 515
https://bio.tools/bioperl - 7
https://bio.tools/biopython - 7
https://bio.tools/arb - 5
https://bio.tools/openms - 5
https://bio.tools/autodock - 5
https://bio.tools/vcftools - 4
https://bio.tools/biosigner - 3
https://bio.tools/biovizbase - 3
https://bio.tools/ensemblvep - 3


### Entries per source

### Coverage across resources

In [45]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT  ?biotools_ID ?source ?application 
#SELECT  ?source (COUNT (?application) AS ?count) 
#SELECT  ?application ?biotools_ID 
WHERE {
    ?application schema:identifier ?biotools_ID .
    #?application ?p schema:SoftwareApplication .
    BIND(
        IF(STRSTARTS ( STR ( ?application ), "https://bioconda" )
        , "bioconda", 
            IF(STRSTARTS ( STR ( ?application ), "https://biocontainers" )
            , "biocontainers", 
                IF(STRSTARTS ( STR ( ?application ), "https://salsa.debian" )
                , "debian", 
                    IF(STRSTARTS ( STR ( ?application ), "https://workflowhub" )
                    , "workflowhub", 
                        IF(STRSTARTS ( STR ( ?application ), "https://bio.tools" )
                        , "biotools", "other")
                        )))) AS ?source)

    FILTER(?source = "bioconda") .
}
#GROUP BY ?source
HAVING STRSTARTS ( STR ( ?biotools_ID ), "https://bio.tools" )
ORDER BY DESC(?biotools_ID)
"""
results = kg.query(query)

for r in results :
    print(f"{r['source']} - {r['biotools_ID']}")


    #FILTER(STRSTARTS(?identifier, "biotools:")) .


bioconda - https://bio.tools/zlibbioc
bioconda - https://bio.tools/zinbwave
bioconda - https://bio.tools/zDB
bioconda - https://bio.tools/yass
bioconda - https://bio.tools/yarn
bioconda - https://bio.tools/yara
bioconda - https://bio.tools/yapsa
bioconda - https://bio.tools/yamss
bioconda - https://bio.tools/yamda
bioconda - https://bio.tools/xvector
bioconda - https://bio.tools/xtandem
bioconda - https://bio.tools/xpore
bioconda - https://bio.tools/xmapbridge
bioconda - https://bio.tools/xhmm
bioconda - https://bio.tools/xde
bioconda - https://bio.tools/xcms
bioconda - https://bio.tools/wtdbg2
bioconda - https://bio.tools/whatshap
bioconda - https://bio.tools/wgd
bioconda - https://bio.tools/wgcna
bioconda - https://bio.tools/weeder
bioconda - https://bio.tools/webbioc
bioconda - https://bio.tools/weaver
bioconda - https://bio.tools/wavcluster
bioconda - https://bio.tools/watermelon
bioconda - https://bio.tools/vsn
bioconda - https://bio.tools/vsearch
bioconda - https://bio.tools/vscl

### Entries with bio.tools IDs by source

In [37]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT  ?source (COUNT (?application) AS ?count) 
#SELECT  ?application ?biotools_ID 
WHERE {
    ?application schema:identifier ?biotools_ID .
    #?application ?p schema:SoftwareApplication .
    BIND(
        IF(STRSTARTS ( STR ( ?application ), "https://bioconda" )
        , "bioconda", 
            IF(STRSTARTS ( STR ( ?application ), "https://biocontainers" )
            , "biocontainers", 
                IF(STRSTARTS ( STR ( ?application ), "https://salsa.debian" )
                , "debian", 
                    IF(STRSTARTS ( STR ( ?application ), "https://workflowhub" )
                    , "workflowhub", 
                        IF(STRSTARTS ( STR ( ?application ), "https://bio.tools" )
                        , "biotools", "other")
                        )))) AS ?source)
}
GROUP BY ?source
HAVING STRSTARTS ( STR ( ?biotools_ID ), "https://bio.tools" )
ORDER BY DESC(?count)
"""
results = kg.query(query)

for r in results :
    print(f"{r['source']} - {r['count']}")


    #FILTER(STRSTARTS(?identifier, "biotools:")) .


biocontainers - 2048
bioconda - 1710


### Entries with NO biotools ID

In [110]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?biotools_ID (COUNT (?application) AS ?count)
WHERE {
    ?application schema:identifier ?biotools_ID .
}
GROUP BY ?biotools_ID
HAVING (COUNT (?application) = 0)
"""
results = kg.query(query)

for r in results :
    print(f"{r['biotools_ID']} - {r['count']}")

### Check bio.tools entries > 4 

In [ ]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX ns1: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?biotools_ID ?application 
WHERE {
    #?application ns1:identifier ?biotools_ID .
    ?application ns1:identifier biotools:bioperl .
}
"""

results = kg.query(query)

for r in results :
    #if r['biotools_ID'] == 'https://bio.tools/bioperl' :
    print(f"{r['biotools_ID']} - {r['application']}")

None - https://bioconda.github.io/recipes/arb-bio
None - https://biocontainers.pro/tools/arb-bio
None - https://biocontainers.pro/tools/arb-bio-devel
None - https://biocontainers.pro/tools/arb-bio-tools
None - https://biocontainers.pro/tools/libarbdb


### Query EDAM ontology

In [91]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX edam: <http://edamontology.org/>
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?edam_class ?edam_label
WHERE {
    ?edam_class rdfs:label ?edam_label .
}
"""
results = edam_kg.query(query)
for r in results :
    print(f"{r['edam_class']} - {r['edam_label']}")

http://edamontology.org/citation - Citation
http://edamontology.org/data_0970 - Citation
http://edamontology.org/created_in - Created in
http://edamontology.org/deprecation_comment - deprecation_comment
http://edamontology.org/documentation - Documentation
http://edamontology.org/example - Example
http://edamontology.org/file_extension - File extension
http://edamontology.org/information_standard - Information standard
http://edamontology.org/is_deprecation_candidate - deprecation_candidate
http://edamontology.org/is_refactor_candidate - refactor_candidate
http://edamontology.org/isdebtag - isdebtag
http://edamontology.org/media_type - Media type
http://edamontology.org/notRecommendedForAnnotation - notRecommendedForAnnotation
http://edamontology.org/obsolete_since - Obsolete since
http://edamontology.org/oldParent - Old parent
http://edamontology.org/oldRelated - Old related
http://edamontology.org/ontology_used - Ontology used
http://edamontology.org/organisation - Organisation
http:

### Get EDAM topics in applications

Is applicationSubcategory necessarily an EDAM topic?

In [194]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX edam: <http://edamontology.org/>
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?edam_class ?edam_label (COUNT (?edam_class) AS ?count)
WHERE {
    ?application schema:applicationSubCategory ?edam_class .
    ?edam_class rdfs:subClassOf+ edam:topic_0003 .
    #            rdfs:label ?edam_label .
    ?edam_class rdfs:label ?edam_label .
}
GROUP BY ?edam_class
ORDER BY DESC(?count)
LIMIT 50
"""
results = kg.query(query)
for r in results :
    print(f"{r['edam_label']} - {r['count']}")

Machine learning - 3678
Molecular interactions, pathways and networks - 3118
Gene expression - 3056
Small molecules - 2960
RNA-Seq - 2886
Sequence analysis - 2434
Sequencing - 2432
Genotype and phenotype - 2081
Proteomics - 2019
Workflows - 1990
Gene transcripts - 1905
Oncology - 1871
Mapping - 1846
Genomics - 1734
Transcription factors and regulatory sites - 1697
Transcriptomics - 1690
Imaging - 1650
Genetic variation - 1650
Pathology - 1538
Cell biology - 1528
Model organisms - 1489
Functional, regulatory and non-coding RNA - 1420
Statistics and probability - 1411
DNA - 1392
Sequence assembly - 1331
DNA polymorphism - 1307
Proteins - 1201
Proteomics experiment - 1178
Genetics - 1147
Protein interactions - 1064
Metagenomics - 1049
Plant biology - 1007
Microarray experiment - 985
RNA - 929
Phylogenetics - 822
Molecular modelling - 818
Structure prediction - 812
Epigenetics - 809
Metabolomics - 802
Bioinformatics - 760
Data management - 748
Sequence sites, features and motifs - 690
GWAS

### Applications related to specific topic and children

In [199]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX edam: <http://edamontology.org/>
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?res ?label (COUNT (?application) AS ?count)
WHERE {
    #?topic schema:identifier edam:topic_0199 .
    ?child rdfs:subClassOf+ edam:topic_0199 .
    FILTER(?res IN (?child, edam:topic_0199))
    ?application schema:applicationSubCategory ?res .
    ?res rdfs:label ?label .
    #?application schema:identifier ?biotools_ID .
}
GROUP BY ?res 
ORDER BY DESC(?count)
"""

#topic_test = "Genetic variation"
#topic_test = "edam:topic_0199"

results = kg.query(query)
#results = kg.query(query, initBindings={'label': Literal(topic_test)})
for r in results :
    print(f"{r['label']} - {r['count']}")

Genetic variation - 6600
DNA polymorphism - 1307
Structural variation - 247
DNA mutation - 95
Copy number variation - 68


### EDAM operations in annotations

In [ ]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX edam: <http://edamontology.org/>
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?edam_class ?edam_label (COUNT (?edam_class) AS ?count)
WHERE {
    ?application schema:featureList ?edam_class .
    ?edam_class rdfs:label ?edam_label .
}
GROUP BY ?edam_class
ORDER BY DESC(?count)
LIMIT 50
"""
results = kg.query(query)
for r in results :
    print(f"{r['edam_label']} - {r['count']}")

### Total number of operations annotated

In [136]:
query = """
PREFIX biotools: <https://bio.tools/>
PREFIX edam: <http://edamontology.org/>
PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT (COUNT(*) AS ?count)
WHERE {
    ?application schema:featureList ?edam_class .
    #?edam_class rdfs:label ?edam_label .
}
"""
results = kg.query(query)
for r in results :
    print(f"{r['count']}")

76865


# Old

## Tests EDAM Control Room

In [42]:
## Get topics that don't have a wikipedia link as a 'seeAlso' attribute

query="""
PREFIX edam: <http://edamontology.org/>
PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?label ?entity WHERE {
    ?entity rdfs:subClassOf+ edam:topic_0003 ;
                rdfs:label ?label .
        FILTER NOT EXISTS {
        ?entity rdfs:seeAlso ?seealso .
        FILTER (regex(str(?seealso), "wikipedia.org", "i"))
       } .
}
"""
#results = kg.query(query)

query = os.path.dirname(".")  + "../queries/no_wikipedia_link_topic.rq"

with open(query, "r") as f:
    query = f.read()
    results = kg.query(query)
f.close()


for r in results :
    print(f"{r['entity']} - {r['label']} ") 


FileNotFoundError: [Errno 2] No such file or directory: '../queries/no_wikipedia_link_topic.rq'

### Links not literals

In [ ]:
query = """
## Checks that all webpage and doi are declared as literal links.

PREFIX obo: <http://purl.obolibrary.org/obo/>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>
PREFIX edam:<http://edamontology.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    
SELECT DISTINCT ?entity ?property ?label ?value WHERE {
    VALUES ?property {
                    <http://edamontology.org/citation>
                    <http://edamontology.org/documentation>
                    <http://edamontology.org/information_standard>
                    <http://edamontology.org/oldParent>
                    <http://edamontology.org/ontology_used>
                    <http://edamontology.org/organisation>
                    <http://www.geneontology.org/formats/oboInOwl#consider>
                    <http://www.geneontology.org/formats/oboInOwl#inSubset>
                    <http://www.geneontology.org/formats/oboInOwl#replacedBy>
                    <http://www.w3.org/2000/01/rdf-schema#seeAlso>
                    <http://www.w3.org/2000/01/rdf-schema#subClassOf>
                    <http://www.w3.org/2000/01/rdf-schema#subPropertyOf>
                    <http://www.w3.org/2002/07/owl#annotatedProperty>
                    <http://www.w3.org/2002/07/owl#inverseOf>
                    <http://www.w3.org/2002/07/owl#disjointWith>
                    <http://www.w3.org/2000/01/rdf-schema#domain>
                    <http://www.w3.org/2000/01/rdf-schema#range>


}

?entity ?property ?value .

FILTER isLiteral(?value) 
?entity rdfs:label ?label .

}
#ORDER BY ?entity
"""

#results = kg.query(query)

query = os.path.dirname(".")  + "../queries/literal_links.rq"

with open(query, "r") as f:
    query = f.read()
    results = kg.query(query)
f.close()


for r in results :
    print(f"{r['entity']} - {r['label']} - {r['property']} - {r['value']} ") 


http://edamontology.org/format_1997 - PHYLIP format - http://edamontology.org/documentation - http://www.bioperl.org/wiki/PHYLIP_multiple_alignment_format 
http://edamontology.org/format_1998 - PHYLIP sequential - http://edamontology.org/documentation - http://www.bioperl.org/wiki/PHYLIP_multiple_alignment_format 
http://edamontology.org/format_3462 - CRAM - http://edamontology.org/documentation - http://www.ebi.ac.uk/ena/software/cram-usage#format_specification http://samtools.github.io/hts-specs/CRAMv2.1.pdf 
http://edamontology.org/format_3484 - ebwt - http://edamontology.org/documentation - https://github.com/BenLangmead/bowtie/blob/master/MANUAL 
http://edamontology.org/format_3485 - RSF - http://edamontology.org/documentation - http://www.molbiol.ox.ac.uk/tutorials/Seqlab_GCG.pdf 
http://edamontology.org/format_3487 - BSML - http://edamontology.org/documentation - http://rothlab.ucdavis.edu/genhelp/chapter_2_using_sequences.html#_Creating_and_Editing_Single_Sequenc 
http://edamon